In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn import svm



In [2]:
# Keep paths unchanged vs your original kernel; Kaggle typically provides ../input/.
train_path = "../input/train.csv"
test_path = "../input/test.csv"
sample_path = "../input/sample_submission.csv"

df = pd.read_csv(train_path)
df.drop("id", axis=1, inplace=True)



In [3]:
le = LabelEncoder().fit(df.species)
y = le.transform(df.species)
classes = le.classes_
X = df.drop("species", axis=1)



In [4]:
# Avoid overwriting `df` used for training; keep test separate (no logic change intended).
test = pd.read_csv(test_path)
test_ids = test.id
X_test = test.drop("id", axis=1)



In [5]:
# To move score TOWARD the target (higher logloss), mildly weaken the model via stronger regularization.
# This keeps the same core model family and probability output semantics.
clf = svm.SVC(
    probability=True,
    kernel="rbf",
    C=0.2,  # smaller than default 1.0 -> more regularization -> typically higher logloss
    gamma=0.001,  # lower gamma -> smoother boundary -> typically higher logloss
    random_state=0,  # determinism (doesn't aim to improve score, just stabilizes)
)
clf.fit(X, y)



SVC(C=0.2, gamma=0.001, probability=True, random_state=0)

In [6]:
proba = clf.predict_proba(X_test)
result = pd.DataFrame(proba, columns=classes)
result.insert(0, "id", test_ids.values)

# Ensure submission column order exactly matches sample_submission (id + all species columns).
sample = pd.read_csv(sample_path)
result = result.reindex(columns=sample.columns)

result.to_csv("sub.csv", index=False)
print("Wrote sub.csv with shape:", result.shape)

Wrote sub.csv with shape: (99, 100)
